In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import ast
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# To remove warnings
import warnings
warnings.filterwarnings('ignore')

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
data = pd.read_csv('/kaggle/input/zomato-bangalore-restaurants/zomato.csv')

In [ ]:
data.head(5)

# Data Description
- ****url****: Contains the url of the restaurant in the zomato website
- ****address****: contains the address of the restaurant in Bengaluru
- ****name****: contains the name of the restaurant
- ****online_order****: whether online ordering is available in the restaurant or not
- ****book_table****: table book option available or not
- ****rate****: contains the overall rating of the restaurant out of 5
- ****votes****: contains total number of rating for the restaurant as of the above mentioned date
- ****phone****: contains the phone number of the restaurant
- ****location****: contains the neighborhood in which the restaurant is located
- ****rest_type****: restaurant type like quick bites or casual dining
- ****dish_liked****: dishes people liked in the restaurant
- ****cuisines****: food styles, separated by comma
- ****approx_cost(for two people)****: contains the approximate cost for meal for two people
- ****reviews_list****: list of tuples containing reviews for the restaurant, each tuple consists of two values, rating and review by the customer
- ****menu_item****: contains list of menus available in the restaurant
- ****listed_in(type)****: type of meal
- ****listed_in(city)****: contains the neighborhood in which the restaurant is listed

In [ ]:
data.info()

# Missing data Information
- ****rate****: 43942 non-null / ****15.03%**** missing data - This coulmn gives the rating given to a restaurant. I think ****ignoring this column**** will be best as imputing it with another value might undervalue or overvalue the rating of a restaurant.
- ****phone****: 50509 non-null / ****2.33%**** missing data - Ignore, the missing data is ****less than 10%****
- ****location****: 51696 non-null / ****0.04%**** missing data - Ignore, the missing data is ****less than 10%****
- ****rest_type****: 51490 non-null / ****0.43%**** misssing data - Ignore, the missing data is ****less than 10%****
- ****dish_liked****: 23639 non-null / ****54.29%**** missing data - This column gives the most liked dish of the restaurant. I can't impute this field as each restaurant is unique and serves different food. ****Ignoring this column**** as well.
- ****cuisines****: 51672 non-null / ****0.08%**** missing data - Ignore, the missing data is ****less than 10%****
- ****approx_cost(for two people)****: 51371 non-null / ****0.67%**** missing data - Ignore, the missing data is ****less than 10%****

# Formatting Data

In [ ]:
# Coverting Object to Numeric and formatting (Run this cell only one time only)
data["rate"] = pd.to_numeric(data["rate"].str.split('/').str[0], errors='coerce')
data["approx_cost(for two people)"] = pd.to_numeric(data["approx_cost(for two people)"], errors='coerce')

In [ ]:
# Formatting phone column (Run this cell only one time only)
data["phone"] = data["phone"].str.split(r'\r\n')
data["phone"] = data["phone"].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)

In [ ]:
# Formatting menu_item column (Run this cell only one time only)
data["menu_item"] = data["menu_item"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else x)
data["menu_item"] = data["menu_item"].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
data["menu_item"] = data["menu_item"].replace('', np.nan) # Replacing empty string with NaN

In [ ]:
# Formatting reviews_list column (Run this cell only one time only)
def clean_reviews(raw_text):
    try:
        # Step 1: Remove outer quotes if present
        if isinstance(raw_text, str) and raw_text.startswith("'") and raw_text.endswith("'"):
            raw_text = raw_text[1:-1]

        # Step 2: Try parsing with ast.literal_eval
        parsed = ast.literal_eval(raw_text)

        entries = []
        for rating, review in parsed:
            # Clean unwanted text
            review = review.replace("RATED\\n", "").replace("RATED\n", "").replace("\\n", "\n").strip()
            # Try fixing unicode junk
            try:
                review = bytes(review, "utf-8").decode("utf-8", "ignore")
            except:
                pass
            entries.append(f"{rating}: {review}")

        return "\n\n".join(entries)

    except Exception as e:
        print("⚠️ Parsing error:", e)
        print("Problematic input snippet:", raw_text[:200])  # Show first 200 chars
        return ""

# Apply to your DataFrame
data['reviews_list'] = data['reviews_list'].apply(clean_reviews)


In [ ]:
data.info()

## After Formatting
- ****menu_item****: 12100 non-null / ****76.6%**** missing data - This column gives menu items so we should be ignoring this but needs to be deleted when using some predictive models.

In [ ]:
data.head(10)

Now the data looks cleaner. Let's perform some visualization :)

# Visualization

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x = "online_order", hue = "book_table", data = data);

There are around 20,000 restaurants not taking online orders and no table booking available. Need to look into why these restaurant are not taking online orders? Is it due to technology illiteracy?

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(y = "listed_in(type)", order = data["listed_in(type)"].value_counts().index, data = data);

In [ ]:
plt.figure(figsize=(8, 8))
sns.countplot(y = "listed_in(city)", order = data["listed_in(city)"].value_counts().index, data = data);

In [ ]:
sns.boxplot(y = "approx_cost(for two people)", data = data)